# 📈 Notebook 04 — Fund Performance Analytics (D4)
**Bluestock Fintech | Day 4**

Formulas: CAGR · Sharpe · Sortino · Alpha · Beta · Max Drawdown · VaR

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = Path('..').resolve()
RAW  = BASE/'data'/'raw'
PROC = BASE/'data'/'processed'

RF_ANNUAL   = 0.065   # RBI repo rate proxy
RF_DAILY    = RF_ANNUAL / 252
TRADING_DAYS = 252

df_nav   = pd.read_csv(RAW/'02_nav_history.csv', parse_dates=['date'])
df_fund  = pd.read_csv(RAW/'01_fund_master.csv').drop_duplicates('amfi_code')
df_bench = pd.read_csv(RAW/'10_benchmark_indices.csv', parse_dates=['date'])
nifty100 = df_bench[df_bench['index_name']=='Nifty100'].sort_values('date').set_index('date')['close_value']
bench_ret = nifty100.pct_change().dropna()
print("Data loaded ✓")

## 1. Daily Returns & CAGR

In [ ]:
results = []
for code, grp in df_nav.groupby('amfi_code'):
    grp = grp.sort_values('date').dropna(subset=['nav'])
    nav = grp['nav'].values
    n   = len(grp)
    ret = pd.Series(nav).pct_change().dropna()

    def cagr(years):
        nd = int(years * TRADING_DAYS)
        if n < nd + 5: return np.nan
        return (nav[-1]/nav[-nd-1]) ** (TRADING_DAYS/nd) - 1

    results.append({
        'amfi_code': code,
        'n_days'   : n,
        'cagr_1yr' : cagr(1),
        'cagr_3yr' : cagr(3),
        'cagr_5yr' : cagr(5),
        'ann_vol'  : ret.std() * np.sqrt(TRADING_DAYS),
        'returns'  : ret,
        'nav_series': pd.Series(nav),
    })

df_cagr = pd.DataFrame([{k:v for k,v in r.items() if k not in ('returns','nav_series')}
                         for r in results])
df_cagr[['cagr_1yr','cagr_3yr','cagr_5yr']] *= 100
df_cagr = df_cagr.round(2)
df_cagr.to_csv(PROC/'cagr_report.csv', index=False)
print(df_cagr.nlargest(10,'cagr_3yr')[['amfi_code','cagr_1yr','cagr_3yr','cagr_5yr']].to_string(index=False))

## 2. Sharpe & Sortino Ratios

In [ ]:
sharpe_rows = []
for r in results:
    ret = r['returns']
    excess = ret - RF_DAILY
    sharpe = (excess.mean() / ret.std() * np.sqrt(TRADING_DAYS)) if ret.std()>0 else np.nan
    dd_ret = ret[ret < 0]
    sortino = (excess.mean() * TRADING_DAYS / (dd_ret.std()*np.sqrt(TRADING_DAYS))) if len(dd_ret)>5 else np.nan
    sharpe_rows.append({'amfi_code':r['amfi_code'],
                        'sharpe_ratio':round(sharpe,2),
                        'sortino_ratio':round(sortino,2) if not np.isnan(sortino) else None,
                        'std_dev_ann_pct':round(r['ann_vol']*100,2)})

df_sharpe = pd.DataFrame(sharpe_rows)
df_sharpe.to_csv(PROC/'sharpe_sortino.csv', index=False)
print("Top 10 by Sharpe:")
print(df_sharpe.nlargest(10,'sharpe_ratio').to_string(index=False))

## 3. Alpha & Beta vs Nifty 100

In [ ]:
ab_rows = []
for r in results:
    ret = r['returns'].copy()
    ret.index = range(len(ret))
    bench_aligned = bench_ret.reindex(df_nav[df_nav['amfi_code']==r['amfi_code']]['date'].iloc[1:]).dropna()
    min_len = min(len(ret), len(bench_aligned))
    if min_len < 60:
        ab_rows.append({'amfi_code':r['amfi_code'],'alpha':None,'beta':None,'te_pct':None})
        continue
    slope, intercept, rval, *_ = stats.linregress(bench_aligned.values[:min_len], ret.values[:min_len])
    alpha  = intercept * TRADING_DAYS
    te     = (ret.values[:min_len] - bench_aligned.values[:min_len]).std() * np.sqrt(TRADING_DAYS)
    ab_rows.append({'amfi_code':r['amfi_code'],
                    'alpha_ann' :round(alpha*100,3),
                    'beta'      :round(slope,3),
                    'r_squared' :round(rval**2,3),
                    'te_pct'    :round(te*100,2)})

df_ab = pd.DataFrame(ab_rows)
df_ab.to_csv(PROC/'alpha_beta.csv', index=False)
print("Alpha-Beta Report:")
print(df_ab.dropna().head(15).to_string(index=False))

## 4. Maximum Drawdown

In [ ]:
mdd_rows = []
for r in results:
    nav = r['nav_series']
    roll_max = nav.cummax()
    drawdown = (nav - roll_max) / roll_max
    mdd = drawdown.min()
    mdd_date_idx = drawdown.idxmin()
    mdd_rows.append({'amfi_code': r['amfi_code'],
                     'max_drawdown_pct': round(mdd*100, 2),
                     'days_in_drawdown': int((drawdown < -0.05).sum())})

df_mdd = pd.DataFrame(mdd_rows)
df_mdd.to_csv(PROC/'max_drawdown.csv', index=False)
print("Worst Max Drawdowns:")
print(df_mdd.nsmallest(10,'max_drawdown_pct').to_string(index=False))

## 5. VaR & CVaR (95%)

In [ ]:
var_rows = []
for r in results:
    ret = r['returns'].dropna()
    if len(ret) < 30: continue
    var95  = float(np.percentile(ret, 5))
    cvar95 = float(ret[ret <= var95].mean())
    var_rows.append({'amfi_code':r['amfi_code'],
                     'var_95_daily_pct' : round(var95*100,3),
                     'cvar_95_daily_pct': round(cvar95*100,3),
                     'var_95_ann_pct'   : round(var95*np.sqrt(TRADING_DAYS)*100,2)})

df_var = pd.DataFrame(var_rows)
df_var.to_csv(PROC/'var_cvar_report.csv', index=False)
print("VaR Report:")
print(df_var.to_string(index=False))

## 6. Fund Scorecard

In [ ]:
df_score = df_cagr[['amfi_code','cagr_3yr']].copy()
df_score = df_score.merge(df_sharpe[['amfi_code','sharpe_ratio']], on='amfi_code', how='left')
df_score = df_score.merge(df_ab[['amfi_code','alpha_ann']], on='amfi_code', how='left')
df_score = df_score.merge(df_mdd[['amfi_code','max_drawdown_pct']], on='amfi_code', how='left')
df_score = df_score.merge(df_fund[['amfi_code','expense_ratio_pct','scheme_name','sub_category']], on='amfi_code', how='left')

df_score['score'] = (
    df_score['cagr_3yr'].rank(pct=True, na_option='bottom') * 30 +
    df_score['sharpe_ratio'].rank(pct=True, na_option='bottom') * 25 +
    df_score['alpha_ann'].rank(pct=True, na_option='bottom') * 20 +
    df_score['max_drawdown_pct'].rank(pct=True, na_option='bottom') * 15 +
    df_score['expense_ratio_pct'].rank(pct=True, ascending=False, na_option='bottom') * 10
)
df_score['rank'] = df_score['score'].rank(ascending=False).astype(int)
df_score = df_score.sort_values('rank')
df_score.to_csv(PROC/'fund_scorecard.csv', index=False)
print("Top 10 Fund Scorecard:")
print(df_score.head(10)[['rank','amfi_code','scheme_name','sub_category',
                          'cagr_3yr','sharpe_ratio','score']].to_string(index=False))

## 7. Benchmark Comparison Chart

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(14,10))

# Chart A: Top 5 funds vs Nifty 100
top5_codes = df_score.head(5)['amfi_code'].tolist()
ax = axes[0]
nifty_sub = df_bench[df_bench['index_name']=='Nifty100'].sort_values('date')
nifty_sub = nifty_sub[nifty_sub['date']>='2022-01-01']
nifty_norm = nifty_sub['close_value'] / nifty_sub['close_value'].iloc[0] * 100
ax.plot(nifty_sub['date'], nifty_norm, 'k--', linewidth=2, label='Nifty 100 (Benchmark)', zorder=5)
for i, code in enumerate(top5_codes):
    sub = df_nav[df_nav['amfi_code']==code].sort_values('date')
    if sub.empty: continue
    norm = sub['nav'] / sub['nav'].iloc[0] * 100
    ax.plot(sub['date'], norm, color=plt.cm.tab10(i/10), linewidth=1.5, label=f"Fund {code}")
ax.set_title('Top 5 Funds vs Nifty 100 Benchmark (2022–2026)', fontsize=13, fontweight='bold')
ax.set_ylabel('Indexed Performance (Base=100)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Chart B: Sharpe ratio bar chart
ax2 = axes[1]
df_s = df_sharpe.dropna(subset=['sharpe_ratio']).nlargest(15,'sharpe_ratio')
colors_bar = ['#2E7D32' if s>1 else '#1565C0' if s>0.5 else '#C62828' for s in df_s['sharpe_ratio']]
bars = ax2.barh(df_s['amfi_code'].astype(str), df_s['sharpe_ratio'], color=colors_bar)
ax2.axvline(1.0, color='green', linestyle='--', alpha=0.7, label='Sharpe=1 (Excellent)')
ax2.axvline(0.5, color='orange', linestyle='--', alpha=0.7, label='Sharpe=0.5 (Good)')
ax2.set_title('Sharpe Ratio by Fund (Rf = 6.5%)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Sharpe Ratio')
ax2.legend(fontsize=8); ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(PROC/'chart_benchmark_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print("Benchmark comparison chart saved ✓")
print("\n✅ D4 Complete — All metrics CSVs saved to data/processed/")